# Tabuleiro de Peças Deslizantes — 8-Puzzle com A*

Resolução do 8-puzzle por busca A* comparando duas heurísticas admissíveis: h1 (peças fora do lugar) e h2 (distância de Manhattan), com contagem de nós expandidos.

**Técnica:** Busca A* com heurísticas admissíveis  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/05-8-puzzle-a-estrela.ipynb)


In [1]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        AGENTE TABULEIRO DE PEÇAS DESLIZANTES — 8-PUZZLE                     ║
║        Projeto Acadêmico de Inteligência Artificial Clássica                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Algoritmos: Busca A* com heurísticas h₁ (Peças Fora do Lugar)             ║
║              e h₂ (Distância de Manhattan)                                  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  Fundamentação: IA-Aula4 (Slides 30–42) | IA-Aula7 (Slides 131–135)       ║
╚══════════════════════════════════════════════════════════════════════════════╝

FUNDAMENTOS TEÓRICOS
════════════════════════════════════════════════════════════════════════════════

1. FORMULAÇÃO DO PROBLEMA (IA-Aula4, Slides 30–32)
   O 8-Puzzle é um benchmark clássico de IA que exige formulação precisa:
   • Estado: posição das 8 peças + posição do espaço vazio numa grade 3×3
   • Estado Inicial: qualquer configuração válida (ex: peças embaralhadas)
   • Estado Objetivo: configuração canônica (1–8 + vazio no canto inf. dir.)
   • Ações: mover o espaço vazio para ESQUERDA, DIREITA, CIMA ou BAIXO
   • Custo: cada movimento tem custo 1 → g(n) = número de movimentos

2. ESPAÇO DE ESTADOS
   O tabuleiro 3×3 possui 9! = 362.880 permutações totais.
   Apenas metade (181.440) são estados acessíveis (alcançáveis a partir do
   objetivo). Estados com número ímpar de inversões NÃO têm solução.

3. REPRESENTAÇÃO DO ESTADO (IA-Aula4, Slides 40–42)
   Usamos uma tupla linear de 9 elementos: (1,2,3,4,5,6,7,8,0)
   Vantagens: imutável (hasheável), comparável, eficiente em memória.
   O valor 0 representa o espaço vazio.
   Mapeamento: índice i → linha i//3, coluna i%3

4. BUSCA A* (IA-Aula7, Slides 131–135)
   Expande pelo menor f(n) = g(n) + h(n):
   • g(n) = custo real acumulado (profundidade)
   • h(n) = estimativa heurística até o objetivo
   Ótimo quando h é admissível: h(n) ≤ h*(n) para todo n.

5. HEURÍSTICAS ADMISSÍVEIS
   h₁: Peças Fora do Lugar — conta peças não no lugar correto
       Simples, mas subestima muito → mais expansões
   h₂: Distância de Manhattan — Σ|xi−xf| + |yi−yf|
       Mais informativa → menos expansões → mais eficiente
   Ambas nunca superestimam o custo real → ambas são admissíveis.

6. COMPLEXIDADE E ESCALABILIDADE
   8-Puzzle:  9 peças  → 9!/2  ≈  181.440  estados  → viável com A*
   15-Puzzle: 16 peças → 16!/2 ≈ 10^13 estados → A* na memória é inviável
   24-Puzzle: 25 peças → 25!/2 ≈ 10^25 estados → requer IDA* ou RBFS
   A explosão combinatória torna buscas cegas inaplicáveis para n > 8.
"""

import heapq
import time
from typing import Optional, Callable
from dataclasses import dataclass, field


# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTES GLOBAIS
# ══════════════════════════════════════════════════════════════════════════════

# Estado objetivo canônico: 1..8 seguido do espaço vazio (0)
OBJETIVO: tuple = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# Operadores: (nome, deslocamento no índice linear)
# O espaço vazio se desloca:
#   ESQUERDA → índice -1  (válido apenas se coluna > 0)
#   DIREITA  → índice +1  (válido apenas se coluna < 2)
#   CIMA     → índice -3  (válido apenas se linha > 0)
#   BAIXO    → índice +3  (válido apenas se linha < 2)
ACOES: list[tuple[str, int]] = [
    ("ESQUERDA", -1),
    ("DIREITA",  +1),
    ("CIMA",     -3),
    ("BAIXO",    +3),
]

# Posições-objetivo de cada peça: peça p está na posição POSICAO_OBJETIVO[p]
# Chave: valor da peça, Valor: (linha, coluna) no estado objetivo
POSICAO_OBJETIVO: dict[int, tuple[int, int]] = {
    val: (i // 3, i % 3)
    for i, val in enumerate(OBJETIVO)
    if val != 0
}


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 1: PUZZLE — REPRESENTAÇÃO E OPERADORES
# ══════════════════════════════════════════════════════════════════════════════

class Puzzle:
    """
    Encapsula a representação do estado e os operadores do 8-Puzzle.

    TEORIA (IA-Aula4, Slides 40–42):
    O estado é representado como uma tupla linear imutável de 9 elementos.
    A imutabilidade permite usar estados como chaves em conjuntos (set)
    e dicionários — essencial para a busca em grafo (closed set).

    Representação:
        (1, 2, 3, 4, 5, 6, 7, 8, 0)
         posição: 0  1  2  3  4  5  6  7  8
         grade:
         [0] [1] [2]      1  2  3
         [3] [4] [5]  →   4  5  6
         [6] [7] [8]      7  8  _
    """

    def __init__(self, estado: tuple):
        if len(estado) != 9 or set(estado) != set(range(9)):
            raise ValueError("Estado inválido: deve conter os valores 0–8 exatamente uma vez.")
        self.estado: tuple = estado

    # ── Propriedades ──────────────────────────────────────────────────────

    @property
    def posicao_vazio(self) -> int:
        """Índice linear do espaço vazio (valor 0)."""
        return self.estado.index(0)

    @property
    def eh_objetivo(self) -> bool:
        """Testa se o estado atual é o estado objetivo."""
        return self.estado == OBJETIVO

    # ── Função Sucessora (IA-Aula4, Slide 17) ────────────────────────────

    def acoes_validas(self) -> list[tuple[str, int]]:
        """
        Retorna as ações aplicáveis no estado atual.

        TEORIA: Nem todos os operadores são válidos em todo estado.
        A validade depende da posição do espaço vazio na grade.
        Ex: se o vazio está na coluna 0, não pode mover para ESQUERDA.
        """
        idx_vazio = self.posicao_vazio
        linha = idx_vazio // 3
        coluna = idx_vazio % 3
        validas = []
        for nome, delta in ACOES:
            novo_idx = idx_vazio + delta
            if nome == "ESQUERDA" and coluna == 0:
                continue
            if nome == "DIREITA"  and coluna == 2:
                continue
            if nome == "CIMA"     and linha  == 0:
                continue
            if nome == "BAIXO"    and linha  == 2:
                continue
            if 0 <= novo_idx < 9:
                validas.append((nome, delta))
        return validas

    def aplicar_acao(self, delta: int) -> 'Puzzle':
        """
        RESULTADO(estado, ação): aplica o operador e retorna novo estado.

        TEORIA (IA-Aula4, Slide 18): O modelo de transição define o estado
        resultante de cada ação. Aqui, trocar o vazio com a peça vizinha
        simula o deslizamento físico da peça.

        Retorna um novo objeto Puzzle (estado imutável — sem mutação).
        """
        idx_vazio = self.posicao_vazio
        novo_idx = idx_vazio + delta
        lista = list(self.estado)
        lista[idx_vazio], lista[novo_idx] = lista[novo_idx], lista[idx_vazio]
        return Puzzle(tuple(lista))

    # ── Representação visual ──────────────────────────────────────────────

    def __str__(self) -> str:
        linhas = []
        for i in range(3):
            linha = []
            for j in range(3):
                val = self.estado[i * 3 + j]
                linha.append("_" if val == 0 else str(val))
            linhas.append(" ".join(linha))
        return "\n".join(linhas)

    def __eq__(self, outro) -> bool:
        return isinstance(outro, Puzzle) and self.estado == outro.estado

    def __hash__(self) -> int:
        return hash(self.estado)


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 2: VALIDAÇÃO DE SOLVABILIDADE
# ══════════════════════════════════════════════════════════════════════════════

def contar_inversoes(estado: tuple) -> int:
    """
    Conta o número de inversões na sequência linear do estado.

    TEORIA: Uma inversão ocorre quando uma peça de valor maior aparece
    antes de uma peça de valor menor, ignorando o espaço vazio (0).
    Um estado do 8-Puzzle é solucionável se e somente se o número de
    inversões for PAR.

    Prova intuitiva: cada movimento horizontal não altera o número de
    inversões; cada movimento vertical altera em ±2 (paridade mantida).
    O objetivo tem 0 inversões (par), logo estados com inversões ímpares
    pertencem ao subespaço inacessível — não têm solução.

    Complexidade: O(n²) onde n = número de peças.
    """
    seq = [v for v in estado if v != 0]
    inversoes = 0
    for i in range(len(seq)):
        for j in range(i + 1, len(seq)):
            if seq[i] > seq[j]:
                inversoes += 1
    return inversoes


def eh_solucionavel(estado: tuple) -> bool:
    """
    Verifica se o estado possui solução.

    Para o 8-Puzzle (grade ímpar), a condição é simples:
    número de inversões deve ser par.
    """
    return contar_inversoes(estado) % 2 == 0


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 3: HEURÍSTICAS
# ══════════════════════════════════════════════════════════════════════════════

class Heuristicas:
    """
    Implementa as heurísticas admissíveis para o 8-Puzzle.

    TEORIA (IA-Aula7, Slides 131–135):
    Uma heurística h(n) é ADMISSÍVEL se nunca superestima o custo real:
        h(n) ≤ h*(n)  para todo estado n
    onde h*(n) é o custo ótimo real de n ao objetivo.

    Admissibilidade garante que A* encontra a solução ótima.

    Uma heurística é CONSISTENTE (monotônica) se:
        h(n) ≤ c(n,a,n') + h(n')
    Consistência implica admissibilidade e garante que A* com busca
    em grafo (closed set) seja ótimo sem precisar reabrir nós.
    """

    @staticmethod
    def pecas_fora_do_lugar(estado: tuple) -> int:
        """
        h₁: Número de peças fora da posição correta.

        ADMISSIBILIDADE: Cada peça fora do lugar precisa de pelo menos
        1 movimento para chegar à posição correta. Logo h₁ ≤ h*.

        LIMITAÇÃO: Subestima muito pois ignora a distância real de cada
        peça ao destino — uma peça no canto oposto contribui com apenas 1,
        mas pode precisar de 4 movimentos.

        Complexidade: O(n) — simples e rápida de calcular.
        """
        return sum(
            1 for i, val in enumerate(estado)
            if val != 0 and val != OBJETIVO[i]
        )

    @staticmethod
    def distancia_manhattan(estado: tuple) -> int:
        """
        h₂: Soma das distâncias de Manhattan de cada peça à sua posição-alvo.

        Distância de Manhattan entre (r1,c1) e (r2,c2):
            d = |r1 - r2| + |c1 - c2|

        ADMISSIBILIDADE: A distância de Manhattan é o mínimo de movimentos
        necessários para deslocar uma peça ao destino em linha reta (sem
        obstáculos). Como obstáculos só aumentam o custo, h₂ ≤ h*.

        CONSISTÊNCIA: Para qualquer movimento, uma peça se aproxima ou
        afasta do objetivo em exatamente 1 unidade Manhattan, portanto
        h₂(n) - h₂(n') ≤ 1 = c(n,a,n'), satisfazendo a desigualdade.

        SUPERIORIDADE sobre h₁: h₂(n) ≥ h₁(n) para todo n, pois
        a distância de cada peça fora do lugar é ≥ 1. Uma heurística
        maior (sem superestimar) sempre gera menos expansões.

        Complexidade: O(n) — percorre todas as peças uma vez.
        """
        total = 0
        for i, val in enumerate(estado):
            if val == 0:
                continue
            linha_atual  = i // 3
            coluna_atual = i % 3
            linha_alvo, coluna_alvo = POSICAO_OBJETIVO[val]
            total += abs(linha_atual - linha_alvo) + abs(coluna_atual - coluna_alvo)
        return total

    @staticmethod
    def zero(estado: tuple) -> int:
        """
        h₀: Heurística nula. Com h=0, A* degenera para UCS/BFS.
        Útil para demonstrar a importância das heurísticas informativas.
        """
        return 0


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 4: NÓ DA ÁRVORE DE BUSCA
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class No:
    """
    Nó na árvore de busca — NÃO confundir com estado do espaço de estados.

    TEORIA (IA-Aula4, Slides 20–22):
    Um estado é uma configuração do mundo. Um nó é uma estrutura de dados
    que representa um ponto na árvore de busca. O mesmo estado pode aparecer
    em múltiplos nós (via caminhos diferentes). O nó contém metadados do
    caminho que levou até ele.

    Campos:
    ─────────────────────────────────────────────────────────────────
    puzzle    : Puzzle → configuração atual do tabuleiro (estado)
    pai       : No     → nó que gerou este (para reconstruir caminho)
    acao      : str    → nome do operador aplicado ("BAIXO", etc.)
    profund   : int    → profundidade na árvore = g(n) (custo=1/mov.)
    h         : int    → valor heurístico h(n)
    f         : int    → f(n) = g(n) + h(n) — critério de ordenação
    ─────────────────────────────────────────────────────────────────
    """
    puzzle:   'Puzzle'
    pai:      Optional['No'] = field(default=None, repr=False)
    acao:     str            = ""
    profund:  int            = 0
    h:        int            = 0
    f:        int            = field(init=False)

    def __post_init__(self):
        self.f = self.profund + self.h   # f = g + h

    def __lt__(self, outro: 'No') -> bool:
        return self.f < outro.f          # necessário para heapq

    def __eq__(self, outro: object) -> bool:
        return isinstance(outro, No) and self.puzzle.estado == outro.puzzle.estado

    def __hash__(self) -> int:
        return hash(self.puzzle.estado)


def reconstruir_caminho(no: No) -> list[tuple[str, 'Puzzle']]:
    """
    Reconstrói a sequência de (ação, estado) do início ao objetivo.

    Segue os ponteiros pai do nó objetivo até a raiz e inverte.
    Complexidade: O(d) onde d é a profundidade da solução.
    """
    caminho = []
    atual = no
    while atual.pai is not None:
        caminho.append((atual.acao, atual.puzzle))
        atual = atual.pai
    return list(reversed(caminho))


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 5: ALGORITMO A*
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ResultadoBusca:
    """Encapsula todos os dados de uma execução da busca."""
    heuristica_nome: str
    solucao:         list[tuple[str, 'Puzzle']]
    custo:           int
    nos_expandidos:  int
    nos_gerados:     int
    tempo_seg:       float
    sucesso:         bool


def busca_a_estrela(
    estado_inicial: tuple,
    heuristica: Callable[[tuple], int],
    nome_heuristica: str = "h",
    verbose: bool = True
) -> ResultadoBusca:
    """
    ════════════════════════════════════════════════════════════════════
    BUSCA A* PARA O 8-PUZZLE
    IA-Aula7, Slides 131–135
    ════════════════════════════════════════════════════════════════════

    PRINCÍPIO: Expande sempre o nó com menor f(n) = g(n) + h(n).
    • g(n) = profundidade = número de movimentos realizados
    • h(n) = estimativa heurística de n ao objetivo
    • f(n) = estimativa do custo total do caminho passando por n

    OTIMALIDADE: Se h é admissível, A* garante solução ótima.
    Prova: O A* nunca expande um nó com f(n) > C* antes do objetivo,
    onde C* é o custo ótimo. Logo, o primeiro nó-objetivo expandido
    tem custo g(n) = C*.

    BUSCA EM GRAFO: Mantemos um conjunto 'explorado' (closed set) para
    evitar reprocessar estados já expandidos. Sem isso, o mesmo estado
    poderia ser expandido infinitas vezes via caminhos diferentes.

    COMPLEXIDADE:
    • Temporal: O(b^d) no pior caso, mas tipicamente muito menor com h₂
    • Espacial: O(b^d) — mantém fronteira e explorado em memória
    • b = fator de ramificação ≈ 2–4 no 8-Puzzle
    • d = profundidade da solução ótima
    """
    t0 = time.perf_counter()

    puzzle_inicial = Puzzle(estado_inicial)
    h0 = heuristica(estado_inicial)
    no_raiz = No(puzzle=puzzle_inicial, h=h0)

    if verbose:
        print(f"\n{'═'*58}")
        print(f"  BUSCA A*  —  Heurística: {nome_heuristica}")
        print(f"{'═'*58}")
        print(f"  Estado inicial:\n{_indent(str(puzzle_inicial))}")
        print(f"  h inicial = {h0}")
        print(f"  f inicial = {no_raiz.f}\n")

    # Fronteira: fila de prioridade mínima por f(n)
    # Tupla: (f, contador, nó) — contador desempata f iguais
    contador = 0
    fronteira: list = [(no_raiz.f, contador, no_raiz)]

    # Melhor f conhecido para cada estado na fronteira
    melhor_f: dict[tuple, int] = {estado_inicial: no_raiz.f}

    # Conjunto explorado (closed set): estados já expandidos
    explorado: set[tuple] = set()

    nos_expandidos = 0
    nos_gerados = 1

    while fronteira:
        _, _, no_atual = heapq.heappop(fronteira)

        # Lazy deletion: ignora se já foi explorado com f menor
        if no_atual.puzzle.estado in explorado:
            continue

        nos_expandidos += 1

        if verbose and nos_expandidos <= 20:
            print(f"  #{nos_expandidos:3d}  Expandindo [{no_atual.acao or 'INÍCIO':>10}]"
                  f"  g={no_atual.profund}  h={no_atual.h}  f={no_atual.f}"
                  f"  estado={no_atual.puzzle.estado}")

        # Teste de objetivo
        if no_atual.puzzle.eh_objetivo:
            tf = time.perf_counter()
            solucao = reconstruir_caminho(no_atual)
            if verbose:
                _imprimir_resultado(nome_heuristica, solucao, no_atual.profund,
                                    nos_expandidos, nos_gerados, tf - t0)
            return ResultadoBusca(
                heuristica_nome=nome_heuristica,
                solucao=solucao,
                custo=no_atual.profund,
                nos_expandidos=nos_expandidos,
                nos_gerados=nos_gerados,
                tempo_seg=tf - t0,
                sucesso=True
            )

        explorado.add(no_atual.puzzle.estado)

        # Expande sucessores via operadores válidos
        for nome_acao, delta in no_atual.puzzle.acoes_validas():
            puzzle_filho = no_atual.puzzle.aplicar_acao(delta)
            estado_filho = puzzle_filho.estado

            if estado_filho in explorado:
                continue

            novo_g = no_atual.profund + 1
            novo_h = heuristica(estado_filho)
            filho = No(
                puzzle=puzzle_filho,
                pai=no_atual,
                acao=nome_acao,
                profund=novo_g,
                h=novo_h
            )
            nos_gerados += 1

            if estado_filho not in melhor_f or filho.f < melhor_f[estado_filho]:
                melhor_f[estado_filho] = filho.f
                contador += 1
                heapq.heappush(fronteira, (filho.f, contador, filho))

    tf = time.perf_counter()
    if verbose:
        print(f"\n  [FALHA] Nenhuma solução encontrada.")
    return ResultadoBusca(
        heuristica_nome=nome_heuristica,
        solucao=[],
        custo=-1,
        nos_expandidos=nos_expandidos,
        nos_gerados=nos_gerados,
        tempo_seg=tf - t0,
        sucesso=False
    )


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 6: IDA* (ITERATIVE DEEPENING A*)
# ══════════════════════════════════════════════════════════════════════════════

def busca_ida_estrela(
    estado_inicial: tuple,
    heuristica: Callable[[tuple], int],
    nome_heuristica: str = "h",
    verbose: bool = True
) -> ResultadoBusca:
    """
    ════════════════════════════════════════════════════════════════════
    IDA* — Iterative Deepening A*
    ════════════════════════════════════════════════════════════════════

    MOTIVAÇÃO: A* mantém toda a fronteira em memória → O(b^d) de espaço.
    Para o 15-Puzzle isso seria inviável. IDA* usa aprofundamento iterativo
    com limite em f(n), consumindo apenas O(d) de memória.

    ALGORITMO:
    • Limite inicial = h(estado_inicial)
    • DFS recursiva: expande apenas nós com f ≤ limite
    • Se não encontra solução: novo limite = mínimo f que excedeu o limite
    • Repete até encontrar ou esgotar possibilidades

    TRADE-OFF: Gera mais nós que A* (reexpande estados), mas usa
    memória linear em vez de exponencial — essencial para puzzles maiores.
    """
    t0 = time.perf_counter()
    if verbose:
        print(f"\n{'═'*58}")
        print(f"  IDA*  —  Heurística: {nome_heuristica}")
        print(f"{'═'*58}\n")

    nos_expandidos = [0]
    nos_gerados    = [1]

    def dfs(no: No, limite: int) -> tuple:
        """DFS com poda por f(n) > limite. Retorna (resultado, próximo_limite)."""
        nos_expandidos[0] += 1

        if no.puzzle.eh_objetivo:
            return reconstruir_caminho(no), no.profund

        if no.f > limite:
            return None, no.f   # sinaliza próximo limite mínimo

        proximo_limite = float('inf')

        for nome_acao, delta in no.puzzle.acoes_validas():
            puzzle_filho = no.puzzle.aplicar_acao(delta)

            # Poda: não voltar ao estado do avô (evita ciclo imediato)
            if no.pai and puzzle_filho.estado == no.pai.puzzle.estado:
                continue

            nos_gerados[0] += 1
            novo_g = no.profund + 1
            novo_h = heuristica(puzzle_filho.estado)
            filho = No(
                puzzle=puzzle_filho,
                pai=no,
                acao=nome_acao,
                profund=novo_g,
                h=novo_h
            )

            resultado, val = dfs(filho, limite)
            if resultado is not None:
                return resultado, val
            if val < proximo_limite:
                proximo_limite = val

        return None, proximo_limite

    puzzle_inicial = Puzzle(estado_inicial)
    h0 = heuristica(estado_inicial)
    no_raiz = No(puzzle=puzzle_inicial, h=h0)
    limite = no_raiz.f
    iteracao = 0

    while True:
        iteracao += 1
        if verbose:
            print(f"  Iteração {iteracao}: limite f = {limite}")
        resultado, novo_limite = dfs(no_raiz, limite)

        if resultado is not None:
            tf = time.perf_counter()
            custo = len(resultado)
            if verbose:
                _imprimir_resultado(nome_heuristica, resultado, custo,
                                    nos_expandidos[0], nos_gerados[0], tf - t0)
            return ResultadoBusca(
                heuristica_nome=f"IDA*+{nome_heuristica}",
                solucao=resultado,
                custo=custo,
                nos_expandidos=nos_expandidos[0],
                nos_gerados=nos_gerados[0],
                tempo_seg=tf - t0,
                sucesso=True
            )
        if novo_limite == float('inf'):
            tf = time.perf_counter()
            return ResultadoBusca(
                heuristica_nome=f"IDA*+{nome_heuristica}",
                solucao=[], custo=-1,
                nos_expandidos=nos_expandidos[0],
                nos_gerados=nos_gerados[0],
                tempo_seg=tf - t0,
                sucesso=False
            )
        limite = novo_limite


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 7: AGENTE
# ══════════════════════════════════════════════════════════════════════════════

class Agente:
    """
    Agente de Resolução de Problemas para o 8-Puzzle.

    CLASSIFICAÇÃO: Agente Baseado em Objetivos com Busca Heurística.
    • Percebe: o estado inicial do tabuleiro
    • Raciocina: formula o problema e busca a solução via A*
    • Age: executa a sequência ótima de movimentos
    • Objetivo: atingir o estado canônico (1–8, vazio no canto)

    Diferente de agentes reativos, este agente PLANEJA a solução completa
    antes de executar qualquer movimento — característico de agentes
    deliberativos (IA-Aula4, Slides 14–16).
    """

    def __init__(self):
        self.historico: list[ResultadoBusca] = []

    def resolver(
        self,
        estado_inicial: tuple,
        heuristica: str = "manhattan",
        algoritmo: str = "A*",
        verbose: bool = True
    ) -> ResultadoBusca:
        """
        Resolve o 8-Puzzle a partir do estado inicial.

        Parâmetros:
            estado_inicial : tupla de 9 elementos (0–8), 0 = vazio
            heuristica     : "manhattan" | "pecas" | "zero"
            algoritmo      : "A*" | "IDA*"
            verbose        : exibe log detalhado da busca
        """
        # Validação
        if not eh_solucionavel(estado_inicial):
            inv = contar_inversoes(estado_inicial)
            raise ValueError(
                f"Estado insolucionável! Inversões: {inv} (ímpar).\n"
                f"Este estado pertence ao subespaço inacessível do 8-Puzzle."
            )

        # Seleciona heurística
        mapa_h = {
            "manhattan": (Heuristicas.distancia_manhattan, "Distância de Manhattan (h₂)"),
            "pecas":     (Heuristicas.pecas_fora_do_lugar, "Peças Fora do Lugar (h₁)"),
            "zero":      (Heuristicas.zero,                "Heurística Nula (h₀ = 0)"),
        }
        if heuristica not in mapa_h:
            raise ValueError(f"Heurística desconhecida: {heuristica}. Use: {list(mapa_h)}")
        fn_h, nome_h = mapa_h[heuristica]

        # Executa busca
        if algoritmo.upper() == "A*":
            resultado = busca_a_estrela(estado_inicial, fn_h, nome_h, verbose)
        elif algoritmo.upper() == "IDA*":
            resultado = busca_ida_estrela(estado_inicial, fn_h, nome_h, verbose)
        else:
            raise ValueError(f"Algoritmo desconhecido: {algoritmo}. Use 'A*' ou 'IDA*'.")

        self.historico.append(resultado)
        return resultado

    def comparar_heuristicas(self, estado_inicial: tuple) -> None:
        """
        Executa A* com h₀, h₁ e h₂ e compara desempenho.

        OBJETIVO DIDÁTICO: Demonstrar empiricamente que heurísticas mais
        informativas (h₂ domina h₁ domina h₀) geram menos expansões,
        mantendo a mesma qualidade ótima de solução.
        """
        print(f"\n{'╔' + '═'*56 + '╗'}")
        print(f"║{'COMPARAÇÃO DE HEURÍSTICAS — A*':^56}║")
        print(f"{'╚' + '═'*56 + '╝'}")
        print(f"\n  Estado inicial:\n{_indent(str(Puzzle(estado_inicial)))}\n")

        configs = [
            ("zero",      "h₀ = Nula (A*≡UCS)"),
            ("pecas",     "h₁ = Peças Fora do Lugar"),
            ("manhattan", "h₂ = Distância de Manhattan"),
        ]
        resultados = []
        for chave, label in configs:
            fn_h, nome_h = {
                "zero":      (Heuristicas.zero,                "h₀"),
                "pecas":     (Heuristicas.pecas_fora_do_lugar, "h₁"),
                "manhattan": (Heuristicas.distancia_manhattan, "h₂"),
            }[chave]
            r = busca_a_estrela(estado_inicial, fn_h, nome_h, verbose=False)
            resultados.append((label, r))

        print(f"\n  {'Heurística':<30} {'Custo':>6} {'Expandidos':>11} {'Gerados':>8} {'Tempo(ms)':>10}")
        print(f"  {'─'*67}")
        for label, r in resultados:
            t_ms = r.tempo_seg * 1000
            print(f"  {label:<30} {r.custo:>6} {r.nos_expandidos:>11} "
                  f"{r.nos_gerados:>8} {t_ms:>9.2f}ms")

        # Análise
        _, r0 = resultados[0]
        _, r1 = resultados[1]
        _, r2 = resultados[2]
        print(f"\n  ANÁLISE:")
        print(f"  ✓ Todos encontraram o mesmo custo ótimo: {r2.custo} movimentos.")
        if r0.nos_expandidos > 0:
            print(f"  ✓ h₁ reduziu expansões em {(1 - r1.nos_expandidos/r0.nos_expandidos)*100:.1f}% vs h₀.")
            print(f"  ✓ h₂ reduziu expansões em {(1 - r2.nos_expandidos/r0.nos_expandidos)*100:.1f}% vs h₀.")
        print(f"  ✓ h₂ domina h₁: h₂(n) ≥ h₁(n) para todo n (mais informativa).")
        print(f"  ✓ Ambas são admissíveis → solução ótima garantida.")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 8: UTILITÁRIOS DE SAÍDA
# ══════════════════════════════════════════════════════════════════════════════

def _indent(texto: str, espacos: int = 4) -> str:
    """Indenta um bloco de texto para exibição."""
    pad = " " * espacos
    return "\n".join(pad + linha for linha in texto.splitlines())


def _imprimir_resultado(
    nome_h: str,
    solucao: list,
    custo: int,
    expandidos: int,
    gerados: int,
    tempo: float
) -> None:
    print(f"\n  {'─'*54}")
    print(f"  ✓ SOLUÇÃO ENCONTRADA")
    print(f"  {'─'*54}")
    print(f"  Heurística  : {nome_h}")
    print(f"  Custo (movs): {custo}")
    print(f"  Expandidos  : {expandidos} nós")
    print(f"  Gerados     : {gerados} nós")
    print(f"  Tempo       : {tempo*1000:.2f} ms")
    print(f"  Sequência   : {' → '.join(a for a, _ in solucao)}")
    print(f"  {'─'*54}")


def exibir_solucao_animada(estado_inicial: tuple, solucao: list) -> None:
    """
    Exibe passo a passo a sequência de movimentos da solução.
    Mostra o tabuleiro antes e depois de cada ação.
    """
    print(f"\n{'═'*40}")
    print(f"  ANIMAÇÃO DA SOLUÇÃO")
    print(f"{'═'*40}")

    puzzle_atual = Puzzle(estado_inicial)
    h_atual = Heuristicas.distancia_manhattan(estado_inicial)
    print(f"\n  Passo 0 — Estado Inicial  [h₂={h_atual}]")
    print(_indent(str(puzzle_atual)))

    for passo, (acao, puzzle_novo) in enumerate(solucao, 1):
        h_novo = Heuristicas.distancia_manhattan(puzzle_novo.estado)
        print(f"\n  Passo {passo} — {acao}  [g={passo}  h₂={h_novo}  f={passo+h_novo}]")
        print(_indent(str(puzzle_novo)))
        puzzle_atual = puzzle_novo

    print(f"\n  ✓ OBJETIVO ATINGIDO em {len(solucao)} movimentos.")


# ══════════════════════════════════════════════════════════════════════════════
# MÓDULO 9: PROGRAMA PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║        AGENTE 8-PUZZLE — BUSCA HEURÍSTICA                   ║")
    print("║        A*  ×  IDA*  |  h₁ Peças  ×  h₂ Manhattan           ║")
    print("╚══════════════════════════════════════════════════════════════╝")

    agente = Agente()

    # ── Exemplo 1: caso simples ───────────────────────────────────────────
    print("\n\n[1] CASO SIMPLES — 2 movimentos")
    print("    Estado inicial: peça 5 e 6 trocadas")
    estado_1 = (1, 2, 3, 4, 0, 6, 7, 5, 8)
    print(f"\n  Estado inicial:\n{_indent(str(Puzzle(estado_1)))}")
    print(f"\n  Inversões: {contar_inversoes(estado_1)} (par → solucionável)")

    r1 = agente.resolver(estado_1, heuristica="manhattan", algoritmo="A*", verbose=True)
    if r1.sucesso:
        exibir_solucao_animada(estado_1, r1.solucao)

    # ── Exemplo 2: caso médio ─────────────────────────────────────────────
    print("\n\n[2] CASO MÉDIO — comparação de heurísticas")
    estado_2 = (1, 2, 3, 4, 5, 6, 0, 7, 8)
    print(f"\n  Estado inicial:\n{_indent(str(Puzzle(estado_2)))}")
    agente.comparar_heuristicas(estado_2)

    # ── Exemplo 3: caso desafiador ────────────────────────────────────────
    print("\n\n[3] CASO DESAFIADOR — A* vs IDA*")
    estado_3 = (8, 1, 3, 4, 0, 2, 7, 6, 5)
    print(f"\n  Estado inicial:\n{_indent(str(Puzzle(estado_3)))}")

    print("\n  — A* com Distância de Manhattan —")
    r3a = agente.resolver(estado_3, heuristica="manhattan", algoritmo="A*",  verbose=True)

    print("\n  — IDA* com Distância de Manhattan —")
    r3b = agente.resolver(estado_3, heuristica="manhattan", algoritmo="IDA*", verbose=True)

    # Tabela comparativa A* vs IDA*
    print(f"\n  {'─'*52}")
    print(f"  {'Métrica':<25} {'A*':>10} {'IDA*':>10}")
    print(f"  {'─'*52}")
    print(f"  {'Custo (movimentos)':<25} {r3a.custo:>10} {r3b.custo:>10}")
    print(f"  {'Nós Expandidos':<25} {r3a.nos_expandidos:>10} {r3b.nos_expandidos:>10}")
    print(f"  {'Nós Gerados':<25} {r3a.nos_gerados:>10} {r3b.nos_gerados:>10}")
    print(f"  {'Tempo (ms)':<25} {r3a.tempo_seg*1000:>9.2f}ms {r3b.tempo_seg*1000:>9.2f}ms")
    print(f"  {'─'*52}")
    print(f"  A*   usa O(b^d) de memória — mantém fronteira completa.")
    print(f"  IDA* usa O(d)   de memória — mas pode reexpandir estados.")

    # ── Exemplo 4: verificação de insolvabilidade ─────────────────────────
    print("\n\n[4] VERIFICAÇÃO DE SOLVABILIDADE")
    estado_invalido = (1, 2, 3, 4, 5, 6, 8, 7, 0)   # 7 e 8 trocados → ímpar
    inv = contar_inversoes(estado_invalido)
    print(f"\n  Estado: {estado_invalido}")
    print(f"  Inversões: {inv} ({'PAR → solucionável' if inv % 2 == 0 else 'ÍMPAR → insolucionável'})")
    if not eh_solucionavel(estado_invalido):
        print(f"  Este estado pertence ao subespaço inacessível do 8-Puzzle.")
        print(f"  Nenhum algoritmo pode resolvê-lo — é matematicamente impossível.")

    # ── Resumo teórico ────────────────────────────────────────────────────
    print(f"\n\n{'═'*62}")
    print(f"  RESUMO TEÓRICO")
    print(f"{'═'*62}")
    print("""
  ESPAÇO DE ESTADOS DO 8-PUZZLE
  • 9! = 362.880 permutações totais; apenas 181.440 são acessíveis.
  • Estados com inversões ímpares são matematicamente insolucionáveis.
  • Crescimento exponencial: 8→15→24 peças → inviabilidade de busca cega.

  HEURÍSTICA h₁ — PEÇAS FORA DO LUGAR
  • Admissível: cada peça fora do lugar precisa de ≥ 1 movimento.
  • Simples de calcular, mas subestima muito → mais expansões.

  HEURÍSTICA h₂ — DISTÂNCIA DE MANHATTAN
  • Admissível: dist. Manhattan ≤ custo real (sem interferência).
  • Consistente: satisfaz desigualdade triangular → ótima com grafo.
  • Domina h₁: h₂(n) ≥ h₁(n) → menos expansões, mesma otimalidade.

  BUSCA A*
  • Expande por f(n)=g(n)+h(n); ótima com h admissível.
  • Usa O(b^d) de memória — limitante para puzzles maiores.

  IDA* (Iterative Deepening A*)
  • Mesmo princípio do A*, mas memória O(d) — viável para 15-Puzzle.
  • Trade-off: reexpande nós, mas escala para problemas maiores.

  ADMISSIBILIDADE → OTIMALIDADE
  • h admissível garante que A* nunca descarta o caminho ótimo.
  • Prova: f(n) ≤ C* para nós no caminho ótimo → eles são expandidos
    antes de qualquer nó com custo > C*.
    """)
    print(f"{'═'*62}")
    print(f"  Projeto concluído.")
    print(f"{'═'*62}\n")


if __name__ == "__main__":
    main()

╔══════════════════════════════════════════════════════════════╗
║        AGENTE 8-PUZZLE — BUSCA HEURÍSTICA                   ║
║        A*  ×  IDA*  |  h₁ Peças  ×  h₂ Manhattan           ║
╚══════════════════════════════════════════════════════════════╝


[1] CASO SIMPLES — 2 movimentos
    Estado inicial: peça 5 e 6 trocadas

  Estado inicial:
    1 2 3
    4 _ 6
    7 5 8

  Inversões: 2 (par → solucionável)

══════════════════════════════════════════════════════════
  BUSCA A*  —  Heurística: Distância de Manhattan (h₂)
══════════════════════════════════════════════════════════
  Estado inicial:
    1 2 3
    4 _ 6
    7 5 8
  h inicial = 2
  f inicial = 2

  #  1  Expandindo [    INÍCIO]  g=0  h=2  f=2  estado=(1, 2, 3, 4, 0, 6, 7, 5, 8)
  #  2  Expandindo [     BAIXO]  g=1  h=1  f=2  estado=(1, 2, 3, 4, 5, 6, 7, 0, 8)
  #  3  Expandindo [   DIREITA]  g=2  h=0  f=2  estado=(1, 2, 3, 4, 5, 6, 7, 8, 0)

  ──────────────────────────────────────────────────────
  ✓ SOLUÇÃO ENCONTRAD